In [10]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, regularizers
import matplotlib.pyplot as plt

# ================= CONFIGURAZIONE =================
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# Percorsi (Devono puntare dove l'Autoencoder ha salvato i dati)
INPUT_PATH = "processed_step2_triple_head"
MODEL_PATH = "saved_models_dynamics_transformer" # Cartella dedicata ai Transformer
os.makedirs(MODEL_PATH, exist_ok=True)

# 1. Caricamento Dati
print("📥 Caricamento dataset...")
try:
    train_raw = pd.read_csv(os.path.join(INPUT_PATH, "train_dataset.csv")).values.astype(np.float32)
    val_raw   = pd.read_csv(os.path.join(INPUT_PATH, "val_dataset.csv")).values.astype(np.float32)
    test_raw  = pd.read_csv(os.path.join(INPUT_PATH, "test_dataset.csv")).values.astype(np.float32)
    LATENT_DIM = train_raw.shape[1]
    print(f"✅ Dati caricati! Latent Dim: {LATENT_DIM}")
except FileNotFoundError:
    print("❌ ERRORE: File non trovati. Controlla il percorso o riesegui lo Step 2.")

# 2. Funzione Windowing (DELTA LEARNING)
# Questa funzione prepara i dati insegnando alla rete a predire solo la DIFFERENZA
def create_sequences(data, seq_length):
    xs, ys = [], []
    for i in range(len(data) - seq_length):
        # Input: Finestra temporale
        x_window = data[i : i + seq_length]
        
        # Target: IL VALORE ASSOLUTO FUTURO (t+1)
        # La rete calcolerà internamente: Output = Ultimo_Input + Delta
        # E confronterà questo Output con il Target Assoluto qui sotto.
        target_val = data[i + seq_length] 
        
        xs.append(x_window)
        ys.append(target_val) 
        
    return np.asarray(xs, dtype=np.float32), np.asarray(ys, dtype=np.float32)

📥 Caricamento dataset...
✅ Dati caricati! Latent Dim: 128


In [11]:
# === CLASSE 1: POSITIONAL ENCODING (Per dare il senso del tempo) ===
class PositionalEmbedding(layers.Layer):
    def __init__(self, seq_len, d_model):
        super().__init__()
        self.d_model = d_model
        self.pos_emb = layers.Embedding(input_dim=seq_len, output_dim=d_model)

    def call(self, x):
        seq_len = tf.shape(x)[1]
        positions = tf.range(start=0, limit=seq_len, delta=1)
        # Somma l'input (dati) con l'embedding (posizione temporale)
        return x + self.pos_emb(positions)

# === CLASSE 2: COSTRUTTORE MODELLO TRANSFORMER ===
def build_transformer_model(seq_len, latent_dim, cfg):
    inputs = layers.Input(shape=(seq_len, latent_dim))
    
    # 1. Input Norm
    x = layers.BatchNormalization(name="Input_Norm")(inputs)
    
    # 2. Positional Encoding
    x = PositionalEmbedding(seq_len, latent_dim)(x)
    
    # 3. Transformer Block (Attention + Feed Forward)
    # MultiHeadAttention: Guarda il passato da diverse prospettive
    attn_output = layers.MultiHeadAttention(num_heads=cfg['n_heads'], 
                                            key_dim=cfg['key_dim'])(x, x)
    x = layers.Add()([x, attn_output])  # Skip Connection 1
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    
    # Feed Forward Network
    ffn = models.Sequential([
        layers.Dense(cfg['ff_dim'], activation="swish"),
        layers.Dropout(cfg['dropout']),
        layers.Dense(latent_dim),
    ])
    ffn_output = ffn(x)
    x = layers.Add()([x, ffn_output])   # Skip Connection 2
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    
    # 4. Global Pooling (Da sequenza a vettore singolo)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(cfg['dropout'])(x)
    
    # 5. Delta Prediction Head
    delta = layers.Dense(latent_dim, activation="linear", name="delta_output")(x)
    
    # 6. RESIDUAL CONNECTION FISICA (Cruciale!)
    # Output = Ultimo_Input + Delta_Predetto
    last_frame = layers.Lambda(lambda t: t[:, -1, :])(inputs)
    outputs = layers.Add(name="final_prediction")([last_frame, delta])
    
    model = models.Model(inputs, outputs, name=cfg['name'])
    
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=cfg['lr']), 
                  loss='mse', metrics=['mae'])
    return model

# === HELPER TRAINING ===
def train_transformer_snippet(cfg):
    print(f"\n🤖 AVVIO CONFIGURAZIONE: {cfg['name']} (Heads={cfg['n_heads']})")
    
    # Prepara i dati (Ricalcola in base alla window della config)
    X_t, y_t = create_sequences(train_raw, cfg['seq_len'])
    X_v, y_v = create_sequences(val_raw, cfg['seq_len'])
    X_test, y_test = create_sequences(test_raw, cfg['seq_len'])
    
    # Costruisci modello
    model = build_transformer_model(cfg['seq_len'], LATENT_DIM, cfg)
    
    # Setup salvataggio
    run_dir = os.path.join(MODEL_PATH, cfg['name'])
    os.makedirs(run_dir, exist_ok=True)
    
    cb = [
        callbacks.EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5),
        callbacks.ModelCheckpoint(os.path.join(run_dir, 'best_model.keras'), save_best_only=True)
    ]
    
    # Train
    history = model.fit(
        X_t, y_t, validation_data=(X_v, y_v),
        epochs=100, batch_size=cfg['batch_size'],
        callbacks=cb, verbose=1
    )
    
    # Stampa risultato finale
    val_loss = min(history.history['val_loss'])
    print(f"🏁 RISULTATO {cfg['name']} -> Best Val MSE: {val_loss:.5f}")
    return history

In [12]:
# === CONFIG 1: STANDARD TRANSFORMER ===
cfg_trans_std = {
    'name': 'Trans_Standard',
    'seq_len': 10,
    'n_heads': 4,       # 4 Teste di attenzione
    'key_dim': 64,      # Dimensione interna standard
    'ff_dim': 128,      # Feed Forward medio
    'dropout': 0.1,
    'lr': 1e-3,
    'batch_size': 64
}

# Lancia il training
hist_std = train_transformer_snippet(cfg_trans_std)


🤖 AVVIO CONFIGURAZIONE: Trans_Standard (Heads=4)
Epoch 1/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 19s 28ms/step - loss: 0.1445 - mae: 0.2963 - val_loss: 0.1420 - val_mae: 0.3005 - learning_rate: 0.0010
Epoch 2/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 13s 23ms/step - loss: 0.0984 - mae: 0.2486 - val_loss: 0.1184 - val_mae: 0.2736 - learning_rate: 0.0010
Epoch 3/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 13s 23ms/step - loss: 0.0613 - mae: 0.1963 - val_loss: 0.0961 - val_mae: 0.2470 - learning_rate: 0.0010
Epoch 4/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 13s 23ms/step - loss: 0.0540 - mae: 0.1842 - val_loss: 0.0932 - val_mae: 0.2432 - learning_rate: 0.0010
Epoch 5/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 21s 23ms/step - loss: 0.0521 - mae: 0.1810 - val_loss: 0.0915 - val_mae: 0.2410 - learning_rate: 0.0010
Epoch 6/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 13s 23ms/step - loss: 0.0510 - mae: 0.1792 - val_loss: 0.0901 - val_mae: 0.2391 - learning_rate: 0.0010
Epoch 7/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 13s 23ms/step - loss: 0.0503 - mae: 0.1

In [13]:
# === CONFIG 2: COMPLEX TRANSFORMER (Più parametri) ===
cfg_trans_complex = {
    'name': 'Trans_Complex',
    'seq_len': 10,
    'n_heads': 8,       # <--- PIÙ TESTE (Più intelligente?)
    'key_dim': 64,
    'ff_dim': 256,      # <--- PIÙ NEURONI INTERNI
    'dropout': 0.2,     # Più dropout per evitare overfitting
    'lr': 5e-4,         # Learning rate più basso per stabilità
    'batch_size': 64
}

# Lancia il training
hist_complex = train_transformer_snippet(cfg_trans_complex)


🤖 AVVIO CONFIGURAZIONE: Trans_Complex (Heads=8)
Epoch 1/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 26s 40ms/step - loss: 0.1780 - mae: 0.3264 - val_loss: 0.1516 - val_mae: 0.3106 - learning_rate: 5.0000e-04
Epoch 2/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 20s 37ms/step - loss: 0.1059 - mae: 0.2585 - val_loss: 0.1431 - val_mae: 0.3017 - learning_rate: 5.0000e-04
Epoch 3/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 20s 36ms/step - loss: 0.0863 - mae: 0.2328 - val_loss: 0.1100 - val_mae: 0.2643 - learning_rate: 5.0000e-04
Epoch 4/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 20s 37ms/step - loss: 0.0633 - mae: 0.1995 - val_loss: 0.0965 - val_mae: 0.2475 - learning_rate: 5.0000e-04
Epoch 5/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 21s 38ms/step - loss: 0.0567 - mae: 0.1889 - val_loss: 0.0902 - val_mae: 0.2393 - learning_rate: 5.0000e-04
Epoch 6/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 24s 45ms/step - loss: 0.0531 - mae: 0.1827 - val_loss: 0.0848 - val_mae: 0.2319 - learning_rate: 5.0000e-04
Epoch 7/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 23s 43ms/step - 

In [14]:
# === CONFIG 3: LONG WINDOW (Memoria Lunga) ===
cfg_trans_long = {
    'name': 'Trans_LongWindow',
    'seq_len': 20,      # <--- GUARDA 20 STEP INDIETRO
    'n_heads': 4,
    'key_dim': 64,
    'ff_dim': 128,
    'dropout': 0.15,
    'lr': 1e-3,
    'batch_size': 64
}

# Lancia il training
hist_long = train_transformer_snippet(cfg_trans_long)


🤖 AVVIO CONFIGURAZIONE: Trans_LongWindow (Heads=4)
Epoch 1/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 20s 31ms/step - loss: 0.1367 - mae: 0.2898 - val_loss: 0.1497 - val_mae: 0.3085 - learning_rate: 0.0010
Epoch 2/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 17s 30ms/step - loss: 0.0916 - mae: 0.2396 - val_loss: 0.1126 - val_mae: 0.2675 - learning_rate: 0.0010
Epoch 3/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 20s 30ms/step - loss: 0.0600 - mae: 0.1942 - val_loss: 0.0972 - val_mae: 0.2485 - learning_rate: 0.0010
Epoch 4/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 21s 30ms/step - loss: 0.0542 - mae: 0.1847 - val_loss: 0.0921 - val_mae: 0.2418 - learning_rate: 0.0010
Epoch 5/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 17s 30ms/step - loss: 0.0518 - mae: 0.1805 - val_loss: 0.0880 - val_mae: 0.2364 - learning_rate: 0.0010
Epoch 6/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 17s 30ms/step - loss: 0.0498 - mae: 0.1770 - val_loss: 0.0847 - val_mae: 0.2318 - learning_rate: 0.0010
Epoch 7/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 17s 30ms/step - loss: 0.0478 - mae: 0